# Model Checkpointer

> Retrospective optimal checkpoint selection system as described in: [tuning_playbook-checkpointer](https://github.com/google-research/tuning_playbook?tab=readme-ov-file#saving-checkpoints-and-retrospectively-selecting-the-best-checkpoint)


In [1]:
#| default_exp utils.checkpointer

In [2]:
#| hide
from nbdev.showdoc import * 

In [2]:
#| export
import os
import pathlib
from pathlib import Path
import heapq
import torch

In [7]:
#| export
class BaseCheckpointer:
    def __init__(self, cfg, client_id, rank=0):
        self.checkpoint_dir = cfg.server.get('checkpoint_dir', 'checkpoints')
        self.checkpoint_dir = os.path.join(self.checkpoint_dir, cfg.project_name, str(cfg.random_seed), f"client_{client_id}")
        self.client_id = client_id
        self.rank = rank

        os.makedirs(self.checkpoint_dir, exist_ok=True)

    def process_checkpoint(self, state, current_acc, step):
        raise NotImplementedError("Must implement process_checkpoint in subclass")
    
    def get_best_checkpoint_path(self):
        raise NotImplementedError("Must implement get_best_checkpoint_path in subclass")

In [8]:
#| export
class RetrospectiveCheckpointer(BaseCheckpointer):
    def __init__(self, cfg, client_id, rank= 0, n_best=3):
        super().__init__(cfg, client_id, rank)

        self.n_best = n_best
        self.best_checkpoints = []
        self.last_path = os.path.join(self.checkpoint_dir, "last_state.pt")
        
        if self.rank == 0 and not os.path.exists(self.checkpoint_dir):
            os.makedirs(self.checkpoint_dir)

    def process_checkpoint(self, state, current_acc, step):
        """
        state: Your dict containing model, optimizer, etc.
        current_acc: The global validation accuracy.
        step: Current training step.
        """
        if self.rank != 0:
            return

        # 1. Always save as "Last" for resilience
        state['id'] = self.client_id
        torch.save(state, self.last_path)

        # 2. Retrospective "Top N" Logic
        filename = f"best_state_step_{step}_acc_{current_acc:.4f}.pt"
        filepath = os.path.join(self.checkpoint_dir, filename)

        # If we have space in our Top N
        if len(self.best_checkpoints) < self.n_best:
            torch.save(state, filepath)
            heapq.heappush(self.best_checkpoints, (current_acc, filename))
            print(f"[Rank 0] Saved new best: {filename}")

        # If this model is better than the worst of the best
        elif current_acc > self.best_checkpoints[0][0]:
            # Remove the worst performing file
            old_acc, old_filename = heapq.heappop(self.best_checkpoints)
            old_filepath = os.path.join(self.checkpoint_dir, old_filename)
            
            if os.path.exists(old_filepath):
                os.remove(old_filepath)
            
            # Save the new one
            torch.save(state, filepath)
            heapq.heappush(self.best_checkpoints, (current_acc, filename))
            print(f"[Rank 0] Replaced {old_filename} with {filename}")

    def get_best_checkpoint_path(self):
        """Returns the path of the absolute highest accuracy checkpoint."""
        if not self.best_checkpoints:
            return None
        # Max in our heap (which tracks accuracy)
        best_filename = max(self.best_checkpoints, key=lambda x: x[0])[1]
        return os.path.join(self.checkpoint_dir, best_filename)

In [ ]:
#| hide
from omegaconf import OmegaConf

In [ ]:
#| hide
model  = torch.nn.Linear(10, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
state = {
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'epoch': 5,
    'accuracy': 0.85
}
cfg = OmegaConf.create({'server': {'checkpoint_dir': './test_checkpoints'}, 'project_name': 'test', 'random_seed': 42})
checkpointer = RetrospectiveCheckpointer(cfg=cfg, client_id=12, rank=0, n_best=3)

In [10]:
#| hide
checkpointer.process_checkpoint(state, current_acc=0.85, step=4)
checkpointer.process_checkpoint(state, current_acc=0.87, step=5)
checkpointer.process_checkpoint(state, current_acc=0.88, step=6)
checkpointer.process_checkpoint(state, current_acc=0.89, step=7)
checkpointer.process_checkpoint(state, current_acc=0.90, step=8)
checkpointer.process_checkpoint(state, current_acc=0.81, step=9)
checkpointer.process_checkpoint(state, current_acc=0.82, step=10)
checkpointer.process_checkpoint(state, current_acc=0.84, step=11)
checkpointer.process_checkpoint(state, current_acc=0.87, step=12)


[Rank 0] Saved new best: best_state_step_4_acc_0.8500.pt
[Rank 0] Saved new best: best_state_step_5_acc_0.8700.pt
[Rank 0] Saved new best: best_state_step_6_acc_0.8800.pt
[Rank 0] Replaced best_state_step_4_acc_0.8500.pt with best_state_step_7_acc_0.8900.pt
[Rank 0] Replaced best_state_step_5_acc_0.8700.pt with best_state_step_8_acc_0.9000.pt


In [11]:
#| hide
for ckpt in checkpointer.best_checkpoints:
    print(ckpt)

(0.88, 'best_state_step_6_acc_0.8800.pt')
(0.89, 'best_state_step_7_acc_0.8900.pt')
(0.9, 'best_state_step_8_acc_0.9000.pt')


In [12]:
#| hide
for ckpt in checkpointer.best_checkpoints:
    state = torch.load(os.path.join(checkpointer.checkpoint_dir, ckpt[1]))
    print("id:", state['id'], "accuracy:", ckpt[0], "epoch:", state['epoch'])


id: 12 accuracy: 0.88 epoch: 5
id: 12 accuracy: 0.89 epoch: 5
id: 12 accuracy: 0.9 epoch: 5


In [13]:
#| hide
best_ckpt = checkpointer.get_best_checkpoint_path()
print("Best checkpoint path:", best_ckpt)

Best checkpoint path: ./test_checkpoints/test/42/client_12/best_state_step_8_acc_0.9000.pt


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()